In [3]:
import pandas as pd
import numpy as np
import wfdb
import ast
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.preprocessing import MultiLabelBinarizer

In [ ]:
#define consts
DATA_PATH = 'dataset/' 
SAMPLING_RATE = 100 
LEADS = [0, 1, 2]#for Lead 1,2,3

In [ ]:
def load_raw_data(df, path):
    data = []
    for f in df.filename_lr:
        signal, _ = wfdb.rdsamp(path + f)
        data.append(signal[:, LEADS]) 
    return np.array(data)

print("Loading metadata")
Y = pd.read_csv(DATA_PATH + 'ptbxl_database.csv', index_col='ecg_id')
Y.scp_codes = Y.scp_codes.apply(lambda x: ast.literal_eval(x))

print("Loading raw ECG signals")
X = load_raw_data(Y, DATA_PATH)

Loading metadata...
Loading raw ECG signals (this will take a few minutes)...


In [ ]:
#Label Mapping (Superclasses)
agg_df = pd.read_csv(DATA_PATH + 'scp_statements.csv', index_col=0)
agg_df = agg_df[agg_df.diagnostic == 1]

def aggregate_diagnostic(y_dic):
    tmp = []
    for key in y_dic.keys():
        if key in agg_df.index:
            tmp.append(agg_df.loc[key].diagnostic_class)
    return list(set(tmp))

Y['diagnostic_superclass'] = Y.scp_codes.apply(aggregate_diagnostic)

# Filter out no valid diagnostic class
valid_indices = Y['diagnostic_superclass'].apply(lambda x: len(x) > 0)
X = X[valid_indices]
Y = Y[valid_indices]

# Binarize labels for Multi-label classification
mlb = MultiLabelBinarizer()
y_bin = mlb.fit_transform(Y['diagnostic_superclass'])
num_classes = len(mlb.classes_)
print(f"Target classes: {mlb.classes_}")

Target classes detected: ['CD' 'HYP' 'MI' 'NORM' 'STTC']


In [ ]:
test_fold = 10
X_train = X[Y.strat_fold != test_fold]
y_train = y_bin[Y.strat_fold != test_fold]
X_test = X[Y.strat_fold == test_fold]
y_test = y_bin[Y.strat_fold == test_fold]

In [ ]:
model = models.Sequential([
    layers.Input(shape=(X_train.shape[1], len(LEADS))), # Input: (1000, 3)
    
    layers.Conv1D(16, kernel_size=7, strides=2, padding='same', activation='relu'),
    layers.MaxPooling1D(pool_size=2),
    
    layers.SeparableConv1D(32, kernel_size=5, padding='same', activation='relu'),
    layers.MaxPooling1D(pool_size=2),
    
    layers.SeparableConv1D(64, kernel_size=3, padding='same', activation='relu'),
    layers.GlobalAveragePooling1D(), # Crucial: replaces Flatten() to save RAM
    
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.Dense(num_classes, activation='sigmoid') 
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=[tf.keras.metrics.AUC(multi_label=True)])
model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 500, 16)        │           352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 250, 16)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv1d                │ (None, 250, 32)        │           624 │
│ (SeparableConv1D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 125, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv1d_1              │ (None, 125, 64)        │         2,208 │
│ (SeparableConv1D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           165 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,429 (21.21 KB)

 Trainable params: 5,429 (21.21 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
print("Starting training...")
model.fit(X_train, y_train, epochs=15, batch_size=64, validation_split=0.1)
print("Quantizing model...")
def representative_data_gen():
    for i in range(200): 
        yield [X_train[i:i+1].astype(np.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_model = converter.convert()

with open('esp32_ecg_model.tflite', 'wb') as f:
    f.write(tflite_model)
    
print("Success. Saved as esp32_ecg_model.tflite")

Starting training...
Epoch 1/15
271/271 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - auc: 0.5179 - loss: 0.5621 - val_auc: 0.6387 - val_loss: 0.5501
Epoch 2/15
271/271 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - auc: 0.6993 - loss: 0.4907 - val_auc: 0.7710 - val_loss: 0.4744
Epoch 3/15
271/271 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - auc: 0.7827 - loss: 0.4407 - val_auc: 0.8075 - val_loss: 0.4409
Epoch 4/15
271/271 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - auc: 0.8107 - loss: 0.4147 - val_auc: 0.8174 - val_loss: 0.4283
Epoch 5/15
271/271 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - auc: 0.8209 - loss: 0.4037 - val_auc: 0.8266 - val_loss: 0.4189
Epoch 6/15
271/271 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - auc: 0.8277 - loss: 0.3953 - val_auc: 0.8338 - val_loss: 0.4095
Epoch 7/15
271/271 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - auc: 0.8354 - loss: 0.3866 - val_auc: 0.8382 - val_loss: 0.4100
Epoch 8/15
271/271 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - auc: 0.8414 - loss: 0.3796 - val_auc: 0.8452 - val_loss: 0.3918
Epoch 9/15
271/271 ━━━━━━━━━━━━━━━━

INFO:tensorflow:Assets written to: C:\Users\Asus\AppData\Local\Temp\tmpteoxisos\assets


Saved artifact at 'C:\Users\Asus\AppData\Local\Temp\tmpteoxisos'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 1000, 3), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 5), dtype=tf.float32, name=None)
Captures:
  2455604025808: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2456379983824: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2456380590992: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2455601330512: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2456377455056: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2456377453712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2455475608976: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2455602347088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2455602335184: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2455602346512: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2456381659

c:\Users\Asus\AppData\Local\Programs\Python\Python313\Lib\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


Success. Saved as esp32_ecg_model.tflite
